# indlæsning af data og libs.

In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [21]:
odr = pd.read_csv("data/age-dependency-ratio-old.csv")
internet = pd.read_csv("data/share-of-individuals-using-the-internet.csv")
tfp = pd.read_csv("data/total-factor-productivity.csv") 
patens = pd.read_csv("data/patent-applications-per-million.csv")
schooling = pd.read_csv("data/average-years-of-schooling-among-adults.csv")

In [22]:
odr = odr.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "odr"
})

internet = internet.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Share of the population using the Internet": "internet_use"
})

tfp = tfp.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Total factor productivity level": "tfp"
})

patens = patens.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Patent applications per million people": "patens"
})

schooling = schooling.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Both genders": "schooling"
})

# lav begrænsing og merging

In [23]:
start = 1999
end = 2020

lagged_odr = odr.copy()

for df in [odr, tfp, internet, patens, schooling]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

odr = odr[(odr["year"] >= start) & (odr["year"] <= end)]
tfp = tfp[(tfp["year"] >= start) & (tfp["year"] <= end)]
internet = internet[(internet["year"] >= start) & (internet["year"] <= end)]
patens = patens[(patens["year"] >= start) & (patens["year"] <= end)]
schooling = schooling[(schooling["year"] >= start) & (schooling["year"] <= end)]

In [24]:
panel = internet[["country", "code", "year", "internet_use"]].merge(
    odr[["code", "year", "odr"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    tfp[["code", "year", "tfp"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    patens[["code", "year", "patens"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    schooling[["code", "year", "schooling"]],
    on=["code", "year"],
    how="inner"
)

panel.head(30)

,country,code,year,internet_use,odr,tfp,patens,schooling
0,Albania,ALB,2011,47.00000,16.544027,1.099752,1.032633,9.800000
1,Albania,ALB,2014,54.30000,17.768805,1.084722,3.605205,10.008679
2,Albania,ALB,2015,56.90000,18.313740,1.045724,5.125777,9.998020
3,Albania,ALB,2016,59.60000,18.861666,0.982690,7.436412,9.987360
4,Albania,ALB,2017,62.40000,19.442000,0.972838,6.041646,9.976700
5,Albania,ALB,2018,65.40000,20.121956,0.960684,5.752122,10.024848
6,Albania,ALB,2019,68.55040,20.942170,0.943900,1.557753,10.072996
7,Angola,AGO,2018,29.00000,5.166748,1.105635,0.191711,5.626794
8,Angola,AGO,2019,32.12940,5.188581,1.079169,0.061775,5.734512
9,Argentina,ARG,1999,3.28000,15.465295,1.133710,24.432878,8.734000


In [25]:
# Vælg de variabler, der skal være til stede
required_vars = ["odr", "tfp", "internet_use", "patens", "schooling"]   # tilføj evt. flere

# Marker rækker hvor alle nødvendige variabler findes
panel["complete_case"] = panel[required_vars].notna().all(axis=1)

# Tæl hvor mange unikke lande der har komplette data i hvert år
countries_per_year = (
    panel.loc[panel["complete_case"]]
    .groupby("year")["code"]
    .nunique()
    .reset_index(name="n_countries_complete")
    .sort_values("year")
)

print(countries_per_year)

    year  n_countries_complete
0   1999                    74
1   2000                    78
2   2001                    75
3   2002                    77
4   2003                    75
5   2004                    75
6   2005                    79
7   2006                    75
8   2007                    82
9   2008                    80
10  2009                    78
11  2010                    80
12  2011                    83
13  2012                    82
14  2013                    86
15  2014                    88
16  2015                    88
17  2016                    91
18  2017                    92
19  2018                    93
20  2019                    89
21  2020                    89
